# 20 · Por malha: a medida dentro de cada malha faz sentido? Malhas de baixa operação contaminam?

> **Regra da disciplina (aula 3):** especificação FOCUS → código gerado pelo assistente → executar → *gate* contra uma resposta já conhecida → aceitar.
> **Quando o gate falha, a restrição que faltou entra em `U`. O código não é remendado.**
>
> Cada passo abaixo tem três partes: a especificação FOCUS enviada ao assistente, o gate declarado **antes** do código (com a fonte da resposta conhecida e da tolerância) e o código que voltou, executado e gateado. O registro de uso de IA é exportado ao final (`registro_uso_ia.csv`).

**Perguntas do autor (16/11, 2h12).** A apuração dentro de cada malha faz sentido? Há malha de baixa operação ou de poucos eventos (por exemplo, a FTC) contaminando o resultado? Falta acrescentar algum atributo?

**Desenho (declarado antes dos resultados).** Mesmo protocolo do notebook 18 (origem móvel, calibração de 2011 a T, conferência de T+1 a T+3, T de 2015 a 2022). (1) AUC por malha, para cada método, com o número de quilômetros atingidos e a mediana de trens por dia. (2) Sensibilidade: retirar as malhas com menos de 50 quilômetros atingidos em 15 anos; manter só as cinco com mais de 200; retirar o quartil de células de menor capacidade vinculada; retirar a FTC. (3) Dois acréscimos avaliados no mesmo protocolo: os atributos ainda não usados que existem na matriz (velocidade do trem vazio, trem-quilômetro por ano, distância média a pátio, produto perigoso) e, como referência, a superfície por especialistas da dissertação (S_esp), que não usa o inventário.

**Gates.** 20.936 células; 2.681 descarrilamentos por via; o universo completo reproduz a AUC de 0,752 do notebook 18.

In [1]:
# === Setup: caminhos, bibliotecas e as três ferramentas (gate, gate_between, log_ia) ===
import os, sys, json, re, unicodedata, datetime as dt
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if os.environ.get("ET261400_BASE"):
    BASE = os.environ["ET261400_BASE"]                # definido pelo notebook 00 (execução em lote) ou pela execução local
elif IN_COLAB and os.path.exists("/content/ET261400/data"):
    BASE = "/content/ET261400"                        # pacote já descompactado na sessão do Colab
elif IN_COLAB and os.path.exists("/content/et261400_colab.zip"):
    import zipfile
    zipfile.ZipFile("/content/et261400_colab.zip").extractall("/content/ET261400")   # pacote enviado pelo painel de arquivos
    BASE = "/content/ET261400"
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = "/content/drive/MyDrive/ET261400"          # pasta do projeto no Drive (ajustar se for outra)
else:
    BASE = os.environ.get("ET261400_BASE", os.path.abspath(".."))   # execução local: raiz do repositório
RAW, PROC = f"{BASE}/data/raw", f"{BASE}/data/processed"
FIG, TAB = f"{BASE}/results/figures", f"{BASE}/results/tables"
for d in (PROC, FIG, TAB):
    os.makedirs(d, exist_ok=True)
print("BASE =", BASE)

_GATES = []          # todo gate executado fica registrado aqui
_LOG_IA = []         # registro de uso de IA: passo, ferramenta, especificação, resultado

def gate(name, got, expected, tol=0.0, source=""):
    """Compara um número calculado com um número que já se tinha antes do código.
    tol vem da fonte (meio passo de arredondamento, faixa publicada), nunca é alargada até passar."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = abs(got - expected) <= tol
    _GATES.append(dict(gate=name, got=got, expected=expected, tol=tol, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, esperado {expected} (tol +/-{tol}) {('| ' + source) if source else ''}")
    return ok

def gate_between(name, got, low, high, source=""):
    """O mesmo, quando a fonte publica uma faixa e não um número."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = low <= got <= high
    _GATES.append(dict(gate=name, got=got, expected=f"[{low}, {high}]", tol=None, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, faixa [{low}, {high}] {('| ' + source) if source else ''}")
    return ok

def log_ia(passo, ferramenta, especificacao, resultado, verificacao):
    """Registro de uso de IA exigido pela disciplina: o que foi pedido, a que ferramenta, o que voltou e como foi verificado."""
    _LOG_IA.append(dict(data=dt.date.today().isoformat(), passo=passo, ferramenta=ferramenta,
                        especificacao=especificacao.strip(), resultado=resultado, verificacao=verificacao))

def exportar_registros(prefixo):
    pd.DataFrame(_GATES).to_csv(f"{TAB}/{prefixo}_gates.csv", index=False)
    pd.DataFrame(_LOG_IA).to_csv(f"{TAB}/{prefixo}_registro_uso_ia.csv", index=False)
    n_ok = sum(g["ok"] for g in _GATES)
    print(f"gates: {n_ok}/{len(_GATES)} PASS | registros de IA: {len(_LOG_IA)} | exportados em {TAB}")

def norm_txt(s):
    """Maiúsculas sem acento, espaços simples: chave de junção para nomes de pátios e estações."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).upper().strip()


BASE = /home/claude


In [2]:
import warnings; warnings.filterwarnings("ignore")
from scipy import sparse
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
C0 = pd.read_csv(f"{PROC}/11_celulas_1km.csv", low_memory=False)
C = C0[~C0["sentido"].astype(str).str.contains("Interromp")].sort_values(["ferrovia", "linha", "km_a"]).reset_index(drop=True); gate("células em operação", len(C), 20936, source="notebook 11")
E = pd.read_csv(f"{PROC}/11_eventos_celulas.csv"); E = E[E["cel_id"].isin(C["cel_id"])].copy(); E["ano"] = E["ano"].astype(int); EV = E[E["via"]]; gate("descarrilamentos por via", len(EV), 2681, source="notebook 11")
N = len(C); idx = pd.Series(np.arange(N), index=C["cel_id"])
def matriz(ev):
    M_ = np.zeros((15, N))
    for k in range(15):
        s = ev[ev["ano"] == 2011 + k].groupby("cel_id").size(); M_[k, idx[s.index].values] = s.values
    return M_
CV_, CA_ = matriz(EV), matriz(E)
def cont(M_, a0, a1): return M_[a0 - 2011:a1 - 2011 + 1].sum(0)
NUM = ["dmin_patio", "dmin_amv", "frac_zona_patio"] + [f"f16p{j}" for j in range(5)] + [f"f16a{j}" for j in range(5)] + ["rmin_geo"] + [f"fr_lt{l}" for l in (300, 400, 500, 600, 800)] + ["rampa_max_pct", "carga_eixo_t", "taxa_dorm", "gab_vert_m", "gab_horiz_m", "cap_vinc", "dias_op", "raio_min_m", "vma_carregado"]
CAT = ["dormente", "perfil_trilho", "fixacao", "sinalizacao", "controle_circ", "bitola", "sentido", "n_linhas"]
EXTRA_NUM, EXTRA_CAT = ["vma_vazio", "tremkm_ano", "dmed_patio"], ["produto_perigoso"]
for c in NUM + EXTRA_NUM + ["S_esp"]: C[c] = pd.to_numeric(C[c], errors="coerce")
for c in CAT + EXTRA_CAT: C[c] = C[c].astype(str).str.strip().replace({"nan": "NA", "": "NA"})
def design(df, num, cat):
    X = df[num].copy(); med = X.median().fillna(0.0)
    for c in num: X[c + "_na"] = X[c].isna().astype(float); X[c] = X[c].fillna(med[c])
    return StandardScaler().fit_transform(pd.concat([X, pd.get_dummies(df[cat], dtype=float)], axis=1).values)
X0 = design(C, NUM, CAT); X1 = design(C, NUM + EXTRA_NUM, CAT + EXTRA_CAT)
FER = np.asarray(C["ferrovia"], dtype=object); FERC = pd.factorize(C["ferrovia"])[0]; LINC = pd.factorize(C["ferrovia"] + "|" + C["linha"].astype(str))[0]; SEG = np.asarray(C["seg_id"].astype(str), dtype=object); KM = ((C["km_a"] + C["km_b"]) / 2).values
ri, ci = [], []
for l in range(LINC.max() + 1):
    i = np.where(LINC == l)[0]; k = KM[i]; M_ = (np.abs(k[:, None] - k[None, :]) <= 20) & ~np.eye(len(i), dtype=bool); a, b = np.where(M_); ri.append(i[a]); ci.append(i[b])
W = sparse.csr_matrix((np.ones(sum(len(a) for a in ri)), (np.concatenate(ri), np.concatenate(ci))), shape=(N, N)); nviz = np.asarray(W.sum(1)).ravel()
def historico(T):
    n = T - 2010; hv = cont(CV_, 2011, T); ha = cont(CA_, 2011, T); b = (hv > 0).astype(float); d = pd.DataFrame({"b": b, "hv": hv})
    return np.column_stack([hv / n, ha / n, b, (W @ b) / np.maximum(nviz, 1), (W @ hv) / np.maximum(nviz, 1) / n, d.groupby(SEG)["b"].transform("mean").values, d.groupby(LINC)["b"].transform("mean").values, d.groupby(FERC)["b"].transform("mean").values, d.groupby(FERC)["hv"].transform("mean").values / n])
cl = np.select([C.dmin_patio.fillna(1e9) <= 300, C.dmin_patio.fillna(1e9) <= 600, C.dmin_patio.fillna(1e9) <= 1200, C.dmin_patio.fillna(1e9) <= 1800], [4, 3, 2, 1], 0); rc = np.select([C.rmin_geo.fillna(1e6) < 500, C.rmin_geo.fillna(1e6) < 1200, C.rmin_geo.fillna(1e6) < 2300], [3, 2, 1], 0); B2 = 0.5 * cl / 4 + 0.5 * rc / 3
def auc(y, s):
    n1 = y.sum(); n0 = len(y) - n1; return float((rankdata(s)[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 >= 5 and n0 >= 5 else np.nan
def auc_dentro(s, y, m=None):
    a, w = [], []
    for f in range(FERC.max() + 1):
        i = (FERC == f) if m is None else ((FERC == f) & m)
        if 5 <= y[i].sum() < i.sum(): a.append(auc(y[i], s[i])); w.append(y[i].sum())
    return float(np.average(a, weights=w)) if a else np.nan
def lr(Xtr, ytr, Xte): return LogisticRegression(C=0.5, max_iter=3000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
NOT = {}
for T in range(2015, 2023):
    ycal = (cont(CV_, 2011, T) > 0).astype(int); y = (cont(CV_, T + 1, T + 3) > 0).astype(int); Htr = historico(T - 3); Hte = historico(T); ytr = (cont(CV_, T - 2, T) > 0).astype(int); st = StandardScaler().fit(Htr)
    NOT[T] = (y, {"Regra do EAR por faixas": B2, "Superfície por especialistas da dissertação (S_esp)": C["S_esp"].fillna(C["S_esp"].median()).values, "EAR calibrado (atributos da via)": lr(X0, ycal, X0), "EAR calibrado + atributos extras": lr(X1, ycal, X1),
                  "EAR calibrado + histórico de acidentes": lr(np.column_stack([X0, st.transform(Htr)]), ytr, np.column_stack([X0, st.transform(Hte)]))})
MET = list(NOT[2015][1])
# 1. Por malha
L = []
tot = (cont(CV_, 2011, 2025) > 0).astype(int)
for f in np.unique(FER):
    i = FER == f; d = dict(malha=f, celulas=int(i.sum()), km_com_descarrilamento_2011_2025=int(tot[i].sum()), descarrilamentos=int(cont(CV_, 2011, 2025)[i].sum()), taxa_pct=round(100 * tot[i].mean(), 1), trens_dia_mediana=float(np.nanmedian(C.loc[i, "cap_vinc"])), celulas_sem_trem=int((C.loc[i, "cap_vinc"].fillna(0) == 0).sum()))
    for m in MET:
        v = [auc(NOT[T][0][i], NOT[T][1][m][i]) for T in NOT]; d[m] = np.nanmean(v) if np.isfinite(v).any() else np.nan; d["conferencias_validas"] = int(np.isfinite(v).sum())
    L.append(d)
PM = pd.DataFrame(L).sort_values("km_com_descarrilamento_2011_2025", ascending=False); PM.to_csv(f"{TAB}/20_por_malha.csv", index=False); display(PM.round(3))
# 2. Sensibilidade: retirar malhas de baixa operação ou de poucos eventos, e células de baixo tráfego
def resumo(m, nome):
    r = []
    for me in MET:
        a = [auc(NOT[T][0][m], NOT[T][1][me][m]) for T in NOT]; d_ = [auc_dentro(NOT[T][1][me], NOT[T][0], m) for T in NOT]; r.append(dict(universo=nome, metodo=me, celulas=int(m.sum()), auc=np.nanmean(a), auc_dentro=np.nanmean(d_)))
    return r
poucos = PM[PM.km_com_descarrilamento_2011_2025 < 50]["malha"].tolist(); q1 = float(C["cap_vinc"].quantile(0.25)); print("malhas com menos de 50 km atingidos em 15 anos:", poucos, "| 1º quartil da capacidade vinculada:", round(q1, 2), "trens por dia")
S = resumo(np.ones(N, bool), "todas as 13 malhas (análise atual)") + resumo(~np.isin(FER, poucos), "sem as malhas com menos de 50 km atingidos") + resumo(np.isin(FER, ["FCA", "FTL", "RMS", "RMP", "RMO"]), "só as cinco malhas com mais de 200 km atingidos") + resumo((C["cap_vinc"].fillna(0) > q1).values, "sem o quartil de menor capacidade vinculada") + resumo(FER != "FTC", "sem a FTC")
SS = pd.DataFrame(S); SS.to_csv(f"{TAB}/20_sensibilidade_malhas.csv", index=False); display(SS.pivot(index="universo", columns="metodo", values="auc").round(3)); display(SS.pivot(index="universo", columns="metodo", values="auc_dentro").round(3))
gate("universo completo reproduz o notebook 18 (AUC do EAR calibrado)", round(float(SS[(SS.universo.str.startswith("todas")) & (SS.metodo == "EAR calibrado (atributos da via)")]["auc"].iloc[0]), 3), 0.752, tol=0.002, source="notebook 18")
log_ia("20 por malha e baixa operação", "Claude", "FOCUS: mesmo protocolo do notebook 18; por malha; sensibilidade a malhas de poucos eventos e a baixo tráfego; atributos extras; superfície por especialistas", "20_por_malha.csv, 20_sensibilidade_malhas.csv", "3 gates")
exportar_registros("20")


[PASS] células em operação: obtido 20936, esperado 20936 (tol +/-0.0) | notebook 11
[PASS] descarrilamentos por via: obtido 2681, esperado 2681 (tol +/-0.0) | notebook 11


,malha,celulas,km_com_descarrilamento_2011_2025,descarrilamentos,taxa_pct,trens_dia_mediana,celulas_sem_trem,Regra do EAR por faixas,conferencias_validas,Superfície por especialistas da dissertação (S_esp),EAR calibrado (atributos da via),EAR calibrado + atributos extras,EAR calibrado + histórico de acidentes
3,FCA,5178,688,853,13.3,2.19,301,0.582,8,0.559,0.659,0.654,0.679
6,FTL,1831,459,640,25.1,2.00,0,0.526,8,0.691,0.703,0.705,0.764
12,RMS,4354,383,450,8.8,0.52,1878,0.619,8,0.597,0.674,0.683,0.705
11,RMP,1179,212,276,18.0,16.62,98,0.653,8,0.615,0.685,0.693,0.737
10,RMO,1603,210,257,13.1,0.36,318,0.544,7,0.460,0.563,0.568,0.656
9,RMN,597,42,50,7.0,9.58,0,0.484,4,0.422,0.455,0.464,0.604
2,EFVM,870,36,50,4.1,17.91,0,0.647,5,0.561,0.581,0.592,0.643
0,EFC,975,35,43,3.6,29.32,0,0.599,3,0.565,0.621,0.629,0.559
7,MRS,1673,25,25,1.5,17.94,11,0.501,2,0.334,0.723,0.722,0.535
4,FNSTN,726,19,21,2.6,3.26,0,0.737,3,0.723,0.755,0.752,0.463


malhas com menos de 50 km atingidos em 15 anos: ['RMN', 'EFVM', 'EFC', 'MRS', 'FNSTN', 'FTC', 'EFPO', 'RMC'] | 1º quartil da capacidade vinculada: 0.62 trens por dia


metodo,EAR calibrado (atributos da via),EAR calibrado + atributos extras,EAR calibrado + histórico de acidentes,Regra do EAR por faixas,Superfície por especialistas da dissertação (S_esp)
universo,,,,,
sem a FTC,0.753,0.752,0.792,0.579,0.695
sem as malhas com menos de 50 km atingidos,0.693,0.691,0.741,0.573,0.612
sem o quartil de menor capacidade vinculada,0.745,0.743,0.788,0.574,0.715
só as cinco malhas com mais de 200 km atingidos,0.693,0.691,0.741,0.573,0.612
todas as 13 malhas (análise atual),0.752,0.752,0.791,0.580,0.695


metodo,EAR calibrado (atributos da via),EAR calibrado + atributos extras,EAR calibrado + histórico de acidentes,Regra do EAR por faixas,Superfície por especialistas da dissertação (S_esp)
universo,,,,,
sem a FTC,0.665,0.666,0.706,0.577,0.589
sem as malhas com menos de 50 km atingidos,0.668,0.669,0.711,0.576,0.590
sem o quartil de menor capacidade vinculada,0.653,0.649,0.688,0.574,0.592
só as cinco malhas com mais de 200 km atingidos,0.668,0.669,0.711,0.576,0.590
todas as 13 malhas (análise atual),0.665,0.666,0.706,0.577,0.589


[PASS] universo completo reproduz o notebook 18 (AUC do EAR calibrado): obtido 0.752, esperado 0.752 (tol +/-0.002) | notebook 18
gates: 3/3 PASS | registros de IA: 1 | exportados em /home/claude/results/tables


---
## Limites

Em malhas com poucos quilômetros atingidos a AUC por malha oscila muito e algumas conferências não são calculáveis (menos de cinco quilômetros atingidos no horizonte). A superfície por especialistas foi construída na dissertação por segmento e é aqui lida por célula.

**Declaração de uso de IA (regra da disciplina, três partes).** Ferramentas: Claude (Anthropic), usado pelo autor como assistente de programação a partir das especificações FOCUS registradas neste notebook, e Gemini do Colab quando indicado. Parte do trabalho: geração do código de cada passo e revisão do texto das células. Responsável pelo conteúdo: o autor, que definiu o problema, as fontes, as especificações, os gates e a verificação, e executou o notebook do zero antes de reportar qualquer número.